# Marts
## Tables finales
### MERGE Delta Lake

**Objectif :** Construire les tables finales consommées par Power BI.

**Principe MERGE :**
Chargement incrémental, seules les lignes nouvelles ou modifiées sont traitées.
Chaque MERGE crée une nouvelle version Delta Lake → Time Travel disponible.

**Pourquoi MERGE over CREATE OR REPLACE ?**

| CREATE OR REPLACE | MERGE Delta Lake |
|---|---|
| Retraite 100% des données | Traite uniquement le delta |
| Écrase l'historique | Préserve chaque version |
| Lent sur gros volumes | Optimisé pour l'incrémental |
| Pas de traçabilité | Audit complet via DESCRIBE HISTORY |

**Tables produites :**

| Table | Clé unique | Lignes | Consommée par |
|---|---|---|---|
| dim_players_info | player_id | 60 | mart_player · mart_physical_condition |
| dim_calendrier_matchs | game_id | 389 | mart_player · mart_game |
| mart_player | game_id + player_id | 4 120 | Power BI · mart_physical_condition |
| mart_game | game_id | 389 | Power BI |
| mart_physical_condition | session_id | 7506 | Power BI |

### Dimensions
Tables de référence | faible volumétrie | mise à jour hebdomadaire

In [0]:
-- ============================================
-- Dim : dim_players_info
-- Référentiel joueurs : biométrie, position, expérience
-- Clé : player_id
-- Fréquence : hebdomadaire (données stables)
-- ============================================
CREATE OR REPLACE TABLE sport_metrics.mart.dim_players_info AS
WITH donnees_source AS (
    SELECT * FROM sport_metrics.staging.team_players_personal_info
),
doublons AS (
    SELECT * FROM donnees_source
    WHERE player_id IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY player_id
        ORDER BY player_id DESC
    ) = 1
)
SELECT * FROM doublons;

In [0]:
-- ============================================
-- Dim : dim_calendrier_matchs
-- Référentiel matchs : dates, saisons, adversaires
-- Clé : game_id
-- Particularité : extraction adversaire depuis Matchup
-- ============================================
CREATE OR REPLACE TABLE sport_metrics.mart.dim_calendrier_matchs AS
WITH donnees_source AS (
    SELECT * FROM sport_metrics.staging.team_games_dataset
)
SELECT
    game_id,
    game_date,
    CASE
        WHEN game_date <= DATE '2020-07-31' THEN '2019-2020'
        WHEN game_date <= DATE '2021-07-31' THEN '2020-2021'
        WHEN game_date <= DATE '2022-07-31' THEN '2021-2022'
        WHEN game_date <= DATE '2023-07-31' THEN '2022-2023'
        ELSE '2023-2024'
    END                                                     AS season,
    EXTRACT(YEAR  FROM game_date)                           AS annee,
    EXTRACT(MONTH FROM game_date)                           AS mois,
    EXTRACT(DAY   FROM game_date)                           AS jour,
    CASE
        WHEN Matchup LIKE '%vs.%' THEN 'Domicile'
        WHEN Matchup LIKE '%@%'   THEN 'Exterieur'
        ELSE NULL
    END                                                     AS place,
    CASE
        WHEN Matchup LIKE '%vs.%' THEN TRIM(SPLIT_PART(Matchup, 'vs.', 2))
        WHEN Matchup LIKE '%@%'   THEN TRIM(SPLIT_PART(Matchup, '@',   2))
        ELSE NULL
    END                                                     AS opponent
FROM donnees_source
WHERE game_id IS NOT NULL;

### mart_player
Table de performance individuelle par match.
MERGE sur `game_id + player_id` clé composite.

In [0]:
-- ============================================
-- mart_player - MERGE
-- Clé composite : game_id + player_id
-- Contient : stats match + biométrie + performance score
-- Performance score = (Pts+Reb+Ast+Stl+Blk) - (TO+PF)
-- ============================================
CREATE OR REPLACE TEMP VIEW source_mart_player AS
WITH sts AS (SELECT * FROM sport_metrics.staging.team_training_sessions),
cm  AS (SELECT * FROM sport_metrics.mart.dim_calendrier_matchs),
pi  AS (SELECT * FROM sport_metrics.mart.dim_players_info),
tps AS (SELECT * FROM sport_metrics.staging.team_players_stats),
last_training AS (
    SELECT * FROM (
        SELECT sts.*,
            ROW_NUMBER() OVER (
                PARTITION BY sts.player_id, sts.next_match_id
                ORDER BY sts.session_date DESC, sts.session_id DESC
            ) AS row_n
        FROM sts
    ) WHERE row_n = 1
)
SELECT
    tps.game_id,
    tps.player_id,
    cm.season,
    pi.player_name,
    pi.age,
    pi.height_cm,
    pi.weight_kg,
    pi.position,
    lt.strength_score                       AS strength_score_last_training,
    cm.place,
    tps.start_position,
    tps.minutes_played,
    tps.points,
    tps.fg_pct,
    tps.fg3_pct,
    tps.ft_pct,
    tps.total_rebounds,
    tps.assists,
    tps.steals,
    tps.blocks,
    tps.turnover,
    tps.player_fault,
    (tps.points + tps.total_rebounds + tps.assists
     + tps.steals + tps.blocks
     - tps.turnover - tps.player_fault)     AS performance_score_match,
    ROUND(
        (tps.points + tps.total_rebounds + tps.assists
         + tps.steals + tps.blocks
         - tps.turnover - tps.player_fault)
        / NULLIF(tps.minutes_played, 0), 2
    )                                       AS performance_score_match_min,
    tps.plus_minus,
    current_timestamp()                     AS updated_at
FROM tps
JOIN cm USING (game_id)
JOIN pi USING (player_id)
JOIN last_training lt
    ON tps.player_id = lt.player_id
    AND tps.game_id = lt.next_match_id;

MERGE INTO sport_metrics.mart.mart_player AS cible
USING source_mart_player AS source
ON cible.game_id = source.game_id
AND cible.player_id = source.player_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

### mart_game
Table de performance collective par match.
MERGE sur `game_id`.
Inclut le Fatigue Index équipe avant chaque match → corrélation fatigue/résultat.

In [0]:
-- ============================================
-- mart_game - MERGE
-- Clé : game_id
-- Inclut Fi_team : moyenne fatigue équipe avant match
-- Permet analyse corrélation fatigue → résultat
-- ============================================
CREATE OR REPLACE TEMP VIEW source_mart_game AS
WITH stg AS (SELECT * FROM sport_metrics.staging.team_games_dataset),
fi  AS (SELECT * FROM sport_metrics.mart.int_fatigue_index),
cm  AS (SELECT * FROM sport_metrics.mart.dim_calendrier_matchs),
fi_equipe AS (
    SELECT
        season,
        next_match_id,
        AVG(fi_before_match) AS fi_team
    FROM fi
    GROUP BY next_match_id, season
),
games AS (
    SELECT
        stg.game_id,
        ROUND(f.fi_team, 2)               AS fi_before_match_team,
        stg.win_loss,
        stg.total_points,
        stg.ecart,
        stg.total_minutes,
        stg.total_field_goal_made,
        stg.total_field_goal_attempt,
        stg.fg_pct,
        stg.total_field_goal_3pts_made,
        stg.total_field_goal_3pts_attempt,
        stg.fg3_pct,
        stg.total_free_throws_made,
        stg.total_free_throws_attempt,
        stg.ft_pct,
        stg.total_offensive_rebounds,
        stg.total_defensive_rebounds,
        stg.total_rebounds,
        stg.total_assists,
        stg.total_steals,
        stg.total_blocks,
        stg.total_turnover,
        stg.total_player_fault,
        stg.win,
        stg.loss,
        stg.win_pct,
        current_timestamp()               AS updated_at
    FROM stg
    INNER JOIN cm ON stg.game_id = cm.game_id
    JOIN fi_equipe f ON f.next_match_id = stg.game_id
)
SELECT * FROM games
QUALIFY ROW_NUMBER() OVER (PARTITION BY game_id ORDER BY game_id) = 1;

MERGE INTO sport_metrics.mart.mart_game AS cible
USING source_mart_game AS source
ON cible.game_id = source.game_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

### mart_physical_condition
Table centrale du projet: croise entraînement + fatigue + performance match.
Alimente le modèle XGBoost prévention blessures.
MERGE sur `session_id`.

In [0]:
-- ============================================
-- mart_physical_condition - MERGE
-- Clé : session_id
-- Table centrale : entraînement + fatigue + match
-- Features ML : fi_avg_7d, fi_avg_28d, training_load_7d/28d
-- Alimente : XGBoost prévention blessures et KMeans clustering profil joueurs
-- ============================================
CREATE OR REPLACE TEMP VIEW source_mart_physical AS
WITH sts AS (SELECT * FROM sport_metrics.staging.team_training_sessions),
fi  AS (SELECT * FROM sport_metrics.mart.int_fatigue_index),
mp  AS (SELECT * FROM sport_metrics.mart.mart_player),
fatigue_stats AS (
    SELECT
        fi.player_id,
        fi.session_id,
        fi.session_date,
        ROUND(AVG(fi.fatigue_index_score) OVER (
            PARTITION BY fi.player_id
            ORDER BY CAST(DATE_FORMAT(fi.session_date, 'yyyyMMdd') AS INT)
            RANGE BETWEEN 7 PRECEDING AND CURRENT ROW
        ), 2)                                           AS fi_avg_7d,
        MAX(fi.fatigue_index_score) OVER (
            PARTITION BY fi.player_id
            ORDER BY CAST(DATE_FORMAT(fi.session_date, 'yyyyMMdd') AS INT)
            RANGE BETWEEN 7 PRECEDING AND CURRENT ROW
        )                                               AS fi_max_7d,
        SUM(sts.duration_min * sts.load_intensity_score) OVER (
            PARTITION BY fi.player_id
            ORDER BY CAST(DATE_FORMAT(fi.session_date, 'yyyyMMdd') AS INT)
            RANGE BETWEEN 7 PRECEDING AND CURRENT ROW
        )                                               AS training_load_7d,
        ROUND(AVG(fi.fatigue_index_score) OVER (
            PARTITION BY fi.player_id
            ORDER BY CAST(DATE_FORMAT(fi.session_date, 'yyyyMMdd') AS INT)
            RANGE BETWEEN 28 PRECEDING AND CURRENT ROW
        ), 2)                                           AS fi_avg_28d,
        MAX(fi.fatigue_index_score) OVER (
            PARTITION BY fi.player_id
            ORDER BY CAST(DATE_FORMAT(fi.session_date, 'yyyyMMdd') AS INT)
            RANGE BETWEEN 28 PRECEDING AND CURRENT ROW
        )                                               AS fi_max_28d,
        SUM(sts.duration_min * sts.load_intensity_score) OVER (
            PARTITION BY fi.player_id
            ORDER BY CAST(DATE_FORMAT(fi.session_date, 'yyyyMMdd') AS INT)
            RANGE BETWEEN 28 PRECEDING AND CURRENT ROW
        )                                               AS training_load_28d
    FROM fi
    JOIN sts USING (session_id)
)
SELECT
    mp.season,
    fi.player_id,
    fi.session_id,
    sts.next_match_id,
    mp.player_name,
    fi.fatigue_index_score                          AS fi_last_training,
    fi.recovery_score                               AS rs_last_training,
    fi.recovery_needed_hours                        AS recovery_needed_last_training,
    fi.fi_before_match,
    sts.focus_level,
    sts.strength_score,
    sts.shooting_accuracy_pct,
    sts.passing_accuracy_pct,
    sts.performance_score,
    sts.load_intensity_score,
    sts.injury_risk,
    fs.fi_avg_7d,
    fs.fi_max_7d,
    fs.training_load_7d,
    fs.fi_avg_28d,
    fs.fi_max_28d,
    fs.training_load_28d,
    mp.place,
    mp.position,
    mp.start_position,
    mp.minutes_played,
    mp.points,
    mp.fg_pct,
    mp.fg3_pct,
    mp.total_rebounds,
    mp.assists,
    mp.steals,
    mp.blocks,
    mp.turnover,
    mp.player_fault,
    mp.performance_score_match,
    mp.performance_score_match_min,
    mp.plus_minus,
    current_timestamp()                             AS updated_at
FROM sts
JOIN mp
    ON mp.game_id = sts.next_match_id
    AND mp.player_id = sts.player_id
JOIN fatigue_stats fs
    ON sts.player_id = fs.player_id
    AND sts.session_id = fs.session_id
    AND sts.session_date = fs.session_date
JOIN fi
    ON sts.player_id = fi.player_id
    AND sts.session_id = fi.session_id
    AND sts.session_date = fi.session_date;

MERGE INTO sport_metrics.mart.mart_physical_condition AS cible
USING source_mart_physical AS source
ON cible.session_id = source.session_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
-- ============================================
-- Vérification finale de toutes les tables mart
-- ============================================
SELECT 'dim_players_info'        AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.mart.dim_players_info
UNION ALL
SELECT 'dim_calendrier_matchs'   AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.mart.dim_calendrier_matchs
UNION ALL
SELECT 'int_fatigue_index'       AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.mart.int_fatigue_index
UNION ALL
SELECT 'mart_player'             AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.mart.mart_player
UNION ALL
SELECT 'mart_game'               AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.mart.mart_game
UNION ALL
SELECT 'mart_physical_condition' AS table_name, COUNT(*) AS nb_lignes FROM sport_metrics.mart.mart_physical_condition
ORDER BY nb_lignes DESC;